---

# `Questões e Contexto`

## 📁 Contexto

No diretório raiz desse documento, existem os três arquivos que serão necessários para a conclusão dessa atividade.

Os dados são fictícios e compreendem uma simulação de um cenário de uma loja de departamentos, para isso temos os arquivos com as seguintes informações:
- **users.csv** → Dados dos clientes/usuários da loja
- **sales.csv** → Dados das vendas
- **products.json** → Dados de cadastro dos produtos

<br><br>

## 📝 Questões

**1.** Declare um novo dataframe que mostre o nome do produto e o valor final da compra.

**2.** Declare um novo dataframe com o valor total gasto por cliente.

**3.** Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o período.

**4.** Declare um novo dataframe com os cinco produtos mais vendidos nos últimos seis meses (considerando período de dados disponível nos arquivos) contendo o nome do produto e a quantidade de produtos vendidos nesse período.

**5.** Calcular a média de faturamento por cliente e o desvio padrão.

**6.** Classificar os clientes em três categorias: silver, gold, platinum;
- **platinum:** clientes que gastaram mais que a média de faturamento por cliente;
- **gold:** clientes que gastaram do menor desvio padrão até a média de faturamento por cliente;
- **silver:** clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;

**7.** Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

In [517]:
### IMPORTANDO BIBLIOTECAS NECESSÁRIAS ###

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.sql.window import Window # biblioteca usada para salvar arquivo parquet

In [3]:
### INSTANCIANDO SPARK ###

spark = SparkSession.builder.appName('test-spark').getOrCreate()

# Respostas:

## Tratamentos necessários / Explorando a base

In [589]:
# Carregando arquivos (products, users, sales)
df_products = spark.read.json("products.json")
df_users = spark.read.csv("users.csv", header=True, inferSchema=True)
df_sales = spark.read.csv("sales.csv", header=True, inferSchema=True)

In [590]:
df_products.show(5, truncate=False)

+-------+----------------+----------+
|price  |product         |product_id|
+-------+----------------+----------+
|$86.50 |Camisa Polo     |1         |
|$112.11|Calça Jeans     |2         |
|$100.19|Vestido de Verão|3         |
|$82.38 |Tênis Esportivo |4         |
|$101.50|Camiseta Básica |5         |
+-------+----------------+----------+
only showing top 5 rows


In [591]:
df_products.count()

50

In [592]:
df_products.dtypes

[('price', 'string'), ('product', 'string'), ('product_id', 'bigint')]

In [593]:
#  Verificando se tem nulos no df products
df_products.select([
    count(
        when(expr(f"try_cast({c} as double)").isNull() if c in ['product_id', 'price'] else col(c).isNull(), True)
    ).alias(c)
    for c in df_products.columns]).show()

+-----+-------+----------+
|price|product|product_id|
+-----+-------+----------+
|   50|      0|         0|
+-----+-------+----------+



In [594]:
df_users.show(5, truncate=False)

+---------+-------------------+-------------------------+------+--------------+----------------+
|client_id|name               |email                    |gender|login         |password        |
+---------+-------------------+-------------------------+------+--------------+----------------+
|1        |Mitch Kilpatrick   |mkilpatrick0@cdc.gov     |M     |mkilpatrick0  |uE9+F7h5*P      |
|2        |Kit Kyncl          |kkyncl1@miitbeian.gov.cn |M     |kkyncl1       |nZ1>gR%L        |
|3        |Marylou Presman    |mpresman2@twitter.com    |F     |mpresman2     |aP2#@KQI        |
|4        |Gilberta Andrieu   |gandrieu3@4shared.com    |M     |gandrieu3     |oN2/1oW(IPSNWwoW|
|5        |Tobiah Boughtflower|tboughtflower4@paypal.com|M     |tboughtflower4|hD7{HV(oHo'C&P. |
+---------+-------------------+-------------------------+------+--------------+----------------+
only showing top 5 rows


In [595]:
df_users.count()

300

In [596]:
df_users.dtypes

[('client_id', 'int'),
 ('name', 'string'),
 ('email', 'string'),
 ('gender', 'string'),
 ('login', 'string'),
 ('password', 'string')]

In [597]:
# Verificando se tem nulos no df users
df_users.select([
    count(
        when(expr(f"try_cast({c} as double)").isNull() if c == 'client_id' else col(c).isNull(), True)
    ).alias(c)
    for c in df_users.columns
]).show()

+---------+----+-----+------+-----+--------+
|client_id|name|email|gender|login|password|
+---------+----+-----+------+-----+--------+
|        0|   0|    0|     0|    0|       0|
+---------+----+-----+------+-----+--------+



In [598]:
df_sales.show(5, truncate=False)

+-------+----------+----------+---------+----+
|sale_id|date      |product_id|client_id|qtde|
+-------+----------+----------+---------+----+
|1      |13/06/2022|30        |223      |2   |
|2      |16/11/2022|23        |175      |2   |
|3      |18/08/2022|29        |184      |2   |
|4      |13/03/2022|31        |194      |1   |
|5      |14/12/2022|13        |221      |2   |
+-------+----------+----------+---------+----+
only showing top 5 rows


In [599]:
df_sales.count()

1000

In [600]:
df_sales.dtypes

[('sale_id', 'int'),
 ('date', 'string'),
 ('product_id', 'int'),
 ('client_id', 'int'),
 ('qtde', 'int')]

In [601]:
# Verificando se tem nulos no df sales
df_sales.select([
    count(
        when(expr(f"try_cast({c} as double)").isNull() if c != 'date' else col(c).isNull(), True)
    ).alias(c)
    for c in df_sales.columns
]).show()

+-------+----+----------+---------+----+
|sale_id|date|product_id|client_id|qtde|
+-------+----+----------+---------+----+
|      0|   0|         0|        0|   0|
+-------+----+----------+---------+----+



In [602]:
# O preço vem como string, limpando para os calculos
df_products = df_products.withColumn("price", regexp_replace(col("price"), "[/$,]", "").cast("double"))

In [603]:
# Removendo duplicados (pelo product_id) e nulos em linhas onde o preço não pôde ser convertido (virou null)
df_products = df_products.dropDuplicates(["product_id"]).dropna(subset=["product_id", "price"])

In [604]:
# Convertendo a coluna date de string para o tipo date real (formato ano/mes/dia - padrão do spark)

df_sales = df_sales.withColumn("date", to_date(col("date"), "dd/MM/yyyy"))

In [605]:
# Verificando se tem nulos no df products para confirmar se a limpeza do '$' funcionou
df_products.select([
    count(
        when(expr(f"try_cast({c} as double)").isNull() if c in ['product_id', 'price'] else col(c).isNull(), True)
    ).alias(c)
    for c in df_products.columns
]).show()

+-----+-------+----------+
|price|product|product_id|
+-----+-------+----------+
|    0|      0|         0|
+-----+-------+----------+



In [606]:
# Removendo usuarios duplicados
df_users = df_users.dropDuplicates(["client_id"])

In [607]:
# Removendo linhas onde o client_id ou name sejam nulos
df_users = df_users.dropna(subset=["client_id", "name"])

In [608]:
# Removendo vendas com quantidades negativas ou nulas
df_sales = df_sales.filter((col("qtde") > 0) & (col("product_id").isNotNull()))

In [609]:
# Removendo duplicados (caso a mesma venda tenha sido computada duas vezes)
df_sales = df_sales.dropDuplicates(["sale_id"])

In [610]:
# Visualizando o resultado e o schema para confirmar as mudanças
df_products.show(5, truncate=False)
df_products.printSchema()

+------+----------------+----------+
|price |product         |product_id|
+------+----------------+----------+
|86.5  |Camisa Polo     |1         |
|112.11|Calça Jeans     |2         |
|100.19|Vestido de Verão|3         |
|82.38 |Tênis Esportivo |4         |
|101.5 |Camiseta Básica |5         |
+------+----------------+----------+
only showing top 5 rows
root
 |-- price: double (nullable = true)
 |-- product: string (nullable = true)
 |-- product_id: long (nullable = true)



In [611]:
df_users.show(5, truncate=False)
df_users.printSchema()

+---------+-------------------+-------------------------+------+--------------+----------------+
|client_id|name               |email                    |gender|login         |password        |
+---------+-------------------+-------------------------+------+--------------+----------------+
|1        |Mitch Kilpatrick   |mkilpatrick0@cdc.gov     |M     |mkilpatrick0  |uE9+F7h5*P      |
|2        |Kit Kyncl          |kkyncl1@miitbeian.gov.cn |M     |kkyncl1       |nZ1>gR%L        |
|3        |Marylou Presman    |mpresman2@twitter.com    |F     |mpresman2     |aP2#@KQI        |
|4        |Gilberta Andrieu   |gandrieu3@4shared.com    |M     |gandrieu3     |oN2/1oW(IPSNWwoW|
|5        |Tobiah Boughtflower|tboughtflower4@paypal.com|M     |tboughtflower4|hD7{HV(oHo'C&P. |
+---------+-------------------+-------------------------+------+--------------+----------------+
only showing top 5 rows
root
 |-- client_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (

In [612]:
df_sales.show(5, truncate=False)
df_sales.printSchema()

+-------+----------+----------+---------+----+
|sale_id|date      |product_id|client_id|qtde|
+-------+----------+----------+---------+----+
|148    |2022-04-17|24        |75       |2   |
|463    |2022-02-13|44        |263      |3   |
|471    |2022-06-03|7         |263      |3   |
|496    |2022-02-07|35        |168      |3   |
|833    |2022-04-04|18        |160      |1   |
+-------+----------+----------+---------+----+
only showing top 5 rows
root
 |-- sale_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- qtde: integer (nullable = true)



## Questão 1

**Enunciado:**

Declare um novo dataframe que mostre o nome do produto e o valor final da compra.

**Resolução:**

In [613]:
df_compras_detalhadas = df_sales.join(df_products, "product_id", "inner")

In [614]:
# Calculando o valor total daquela linha/venda específica (quantidade * preço)
df_compras_detalhadas = df_compras_detalhadas.withColumn("valor_final_compra",
                                                         round(col("qtde") * col("price"), 2))

In [615]:
# Selecionando apenas as colunas relevantes para a visualização
df_resultado_final = df_compras_detalhadas.select(
    "sale_id",
    "product",
    "qtde",
    "price",
    "valor_final_compra")

In [616]:
df_resultado_final.show(10, truncate=False)

+-------+--------------------+----+------+------------------+
|sale_id|product             |qtde|price |valor_final_compra|
+-------+--------------------+----+------+------------------+
|148    |Sapatilhas          |2   |123.33|246.66            |
|463    |Mochila de Lona     |3   |117.62|352.86            |
|471    |Blusa de Manga Longa|3   |100.37|301.11            |
|496    |Vestido Midi        |3   |110.27|330.81            |
|833    |Cinto de Couro      |1   |105.47|105.47            |
|243    |Blusa de Renda      |3   |98.9  |296.7             |
|392    |Bolsa de Couro      |3   |117.15|351.45            |
|540    |Mochila de Lona     |1   |117.62|117.62            |
|623    |Cinto de Couro      |1   |105.47|105.47            |
|737    |Calça Flare         |1   |82.87 |82.87             |
+-------+--------------------+----+------+------------------+
only showing top 10 rows


----------------------------------------------------------------------------------------

- Definindo valor total vendido por produto

In [617]:
df_vendas_com_preco_total = df_sales.join(df_products, "product_id", "inner")

In [618]:
df_vendas_com_preco_total = df_vendas_com_preco_total.withColumn("faturamento_linha", col("qtde") * col("price"))

In [619]:
# Agrupando por nome do produto e somando o faturamento
df_resultado__total = df_vendas_com_preco_total.groupBy("product") \
    .agg(round(sum("faturamento_linha"), 2).alias("faturamento_total")) \
    .orderBy(col("faturamento_total").desc())

In [620]:
df_resultado__total.show(10, truncate=False)

+--------------------+-----------------+
|product             |faturamento_total|
+--------------------+-----------------+
|Mochila de Lona     |7292.44          |
|Blusa de Manga Longa|6825.16          |
|Camisola de Seda    |6423.6           |
|Casaco de Inverno   |5894.0           |
|Saia Plissada       |5733.12          |
|Cinto de Couro      |5695.38          |
|Calça Cintura Alta  |5683.6           |
|Camisa Xadrez       |5648.5           |
|Blusa de Gola Alta  |5291.26          |
|Jaqueta de Couro    |5215.14          |
+--------------------+-----------------+
only showing top 10 rows


## Questão 2

**Enunciado:**

Declare um novo dataframe com o valor total gasto por cliente.

**Resolução:**

In [621]:
# Unindo as vendas com os produtos para obter o preço unitario de cada item vendido
df_vendas_precos = df_sales.join(df_products, "product_id", "inner")

In [622]:
# Calculando o valor de cada linha
df_vendas_precos = df_vendas_precos.withColumn("valor_linha", col("qtde") * col("price"))

In [623]:
# Criando o novo df agrupando por cliente e somando o gasto total
df_gasto_por_cliente = df_vendas_precos.groupBy("client_id") \
    .agg(round(sum("valor_linha"), 2).alias("valor_total_gasto"))

In [624]:
df_gasto_por_cliente.show(5)

+---------+-----------------+
|client_id|valor_total_gasto|
+---------+-----------------+
|      148|           307.88|
|      243|           280.28|
|       31|           1755.6|
|      251|           837.55|
|      137|           624.82|
+---------+-----------------+
only showing top 5 rows


In [625]:
# Unindo com df_users para ver o nome do cliente em vez de apenas o id
df_gasto_por_cliente_com_nome = df_gasto_por_cliente.join(df_users.select("client_id", "name"), "client_id", "inner")

In [626]:
df_gasto_por_cliente_com_nome.orderBy(col("valor_total_gasto").desc()).show(10)

+---------+-----------------+-------------------+
|client_id|valor_total_gasto|               name|
+---------+-----------------+-------------------+
|      131|          2240.51|  Randa Friedenbach|
|      245|          1902.85|    Giuditta Blease|
|       24|          1843.35|       Cher Higford|
|      200|          1795.86|      Keen Juggings|
|       31|           1755.6|      Alfie Pattlel|
|       53|          1751.67|   Pieter Gawthorpe|
|      222|          1713.44|Margarethe Prigmore|
|       19|          1690.69|    Brandie Winning|
|      168|          1616.26|      Newton Nother|
|      210|          1600.66|     Germaine Pagen|
+---------+-----------------+-------------------+
only showing top 10 rows


## Questão 3

**Enunciado:**

Declare um novo dataframe com os cinco melhores clientes contendo o nome, e-mail e o valor gasto em todo o período.

**Resolução:**

In [627]:
# Primeiro pego o preço e depois os dados cadastrais (users)
df_vendas_completas = df_sales.join(df_products, "product_id", "inner") \
                               .join(df_users, "client_id", "inner")

In [628]:
# Calculando o valor de cada item vendido
df_vendas_completas = df_vendas_completas.withColumn(
    "valor_venda", col("qtde") * col("price"))

In [629]:
df_top_5_clientes = df_vendas_completas.groupBy("client_id", "name", "email") \
    .agg(round(sum("valor_venda"), 2).alias("valor_gasto_total")) \
    .orderBy(col("valor_gasto_total").desc()) \
    .limit(5)

In [630]:
df_top_5_clientes.show(truncate=False)

+---------+-----------------+-------------------------+-----------------+
|client_id|name             |email                    |valor_gasto_total|
+---------+-----------------+-------------------------+-----------------+
|131      |Randa Friedenbach|rfriedenbach3m@paypal.com|2240.51          |
|245      |Giuditta Blease  |gblease6s@friendfeed.com |1902.85          |
|24       |Cher Higford     |chigfordn@issuu.com      |1843.35          |
|200      |Keen Juggings    |kjuggings5j@phoca.cz     |1795.86          |
|31       |Alfie Pattlel    |apattlelu@discuz.net     |1755.6           |
+---------+-----------------+-------------------------+-----------------+



## Questão 4

**Enunciado:**

Declare um novo dataframe com os cinco produtos mais vendidos nos últimos seis meses (considerando período de dados disponível nos arquivos) contendo o nome do produto e a quantidade de produtos vendidos nesse período.

**Resolução:**

In [631]:
# Descobrindo qual é a data máxima (última venda)
data_maxima = df_sales.select(max("date")).collect()[0][0]
data_maxima

datetime.date(2022, 12, 30)

In [632]:
# Calculando o limite de 6 meses atrás a partir da data máxima
data_limite = df_sales.select(add_months(max("date"), -6)).collect()[0][0]
data_limite

datetime.date(2022, 6, 30)

In [633]:
# Filtrando as vendas apenas nesse período
df_top_5_produtos_6m = df_sales.filter(col("date") >= data_limite) \
    .join(df_products, "product_id", "inner") \
    .groupBy("product") \
    .agg(sum("qtde").alias("quantidade_vendida")) \
    .orderBy(col("quantidade_vendida").desc()) \
    .limit(5)

In [634]:
df_top_5_produtos_6m.show(truncate=False)

+--------------------+------------------+
|product             |quantidade_vendida|
+--------------------+------------------+
|Macaquinho          |39                |
|Blusa de Manga Longa|34                |
|Blusa de Malha      |29                |
|Casaco de Inverno   |29                |
|Bolsa de Couro      |27                |
+--------------------+------------------+



## Questão 5

**Enunciado:**

Calcular a média de faturamento por cliente e o desvio padrão.  

**Resolução:**

In [635]:
# Calculando o gasto total de cada cliente
df_gastos_clientes = df_sales.join(df_products, "product_id", "inner") \
    .withColumn("valor_linha", col("qtde") * col("price")) \
    .groupBy("client_id") \
    .agg(sum("valor_linha").alias("total_gasto_cliente"))

In [636]:
# Calculando a média e o desvio padrao sobre esse novo DF
df_estatisticas = df_gastos_clientes.select(
    round(avg("total_gasto_cliente"), 2).alias("media_faturamento_cliente"),
    round(stddev("total_gasto_cliente"), 2).alias("desvio_padrao_faturamento"))

In [637]:
df_estatisticas.show()

+-------------------------+-------------------------+
|media_faturamento_cliente|desvio_padrao_faturamento|
+-------------------------+-------------------------+
|                   707.26|                   414.34|
+-------------------------+-------------------------+



## Questão 6

**Enunciado:**

Classificar os clientes em três categorias: silver, gold, platinum

- platinum: clientes que gastaram mais que a média de faturamento por cliente;
- gold: clientes que gastaram do menor desvio padrão até a média de faturamento por cliente;
- silver: clientes que gastaram no máximo a média menos o desvio padrão do faturamento por cliente;


**Resolução:**

In [638]:
# Capturando os valores exatos da questao anterior (faturamento e desvio padrão)
stats = df_estatisticas.collect()[0]
media = stats['media_faturamento_cliente']
desvio = stats['desvio_padrao_faturamento']

In [639]:
media

707.26

In [640]:
desvio

414.34

In [641]:
# Calculando os limites conforme o enunciado
limite_platinum = media
limite_silver = media - desvio

In [642]:
limite_silver

292.92

In [643]:
# Aplicando a classificação
df_classificado = df_gastos_clientes.withColumn("categoria",
    when(col("total_gasto_cliente") > lit(limite_platinum), "platinum")
    .when(col("total_gasto_cliente") <= lit(limite_silver), "silver")
    .otherwise("gold"))

In [644]:
# Exibindo o raking final com nomes
df_final = df_classificado.join(df_users.select("client_id", "name"), "client_id", "inner") \
    .select("name", round(col("total_gasto_cliente"), 2).alias("total_gasto_cliente"), "categoria")

In [645]:
df_final.show()

+-------------------+-------------------+---------+
|               name|total_gasto_cliente|categoria|
+-------------------+-------------------+---------+
|   Mitch Kilpatrick|             244.39|   silver|
|          Kit Kyncl|            1289.51| platinum|
|    Marylou Presman|            1067.31| platinum|
|   Gilberta Andrieu|            1335.23| platinum|
|Tobiah Boughtflower|             224.22|   silver|
|      Prent Clifton|              320.8|     gold|
|   Katinka Mosedill|             513.84|     gold|
|       Aime Cheyney|             337.74|     gold|
|          Zoe Exrol|             341.72|     gold|
|       Pepillo Ryde|            1085.22| platinum|
|       Emeline Liff|              339.3|     gold|
|      Kyrstin Black|              566.4|     gold|
|     Ganny Richards|            1278.64| platinum|
|        Robbie Cady|             653.61|     gold|
|   Findley Kimbling|             220.12|   silver|
|     Edgard Stuckow|             445.02|     gold|
|    Brandie

In [646]:
# Exibindo somente clientes platinum
df_final.filter(col("categoria") == "platinum").show(10)

+------------------+-------------------+---------+
|              name|total_gasto_cliente|categoria|
+------------------+-------------------+---------+
|     Alfie Pattlel|             1755.6| platinum|
|  Paulette Tilburn|             837.55| platinum|
|Woodrow Leadbetter|             791.36| platinum|
|  Pieter Gawthorpe|            1751.67| platinum|
|      Derrek Gaine|             787.08| platinum|
|      Cary Syddall|            1107.99| platinum|
|      Geralda Over|            1260.94| platinum|
|      Cammy Solway|             974.77| platinum|
|  Trenton Phillipp|              851.9| platinum|
|   Emogene Scoines|             708.44| platinum|
+------------------+-------------------+---------+
only showing top 10 rows


In [647]:
# Exibindo somente clientes gold
df_final.filter(col("categoria") == "gold").show(10)

+----------------+-------------------+---------+
|            name|total_gasto_cliente|categoria|
+----------------+-------------------+---------+
| Ignatius Baudic|             307.88|     gold|
|   Darren Warman|             624.82|     gold|
|    Kylie Bolton|             688.92|     gold|
|    Tracy Lorand|             538.87|     gold|
|  Thekla Cuniffe|             374.04|     gold|
|Corbett St. Paul|             638.25|     gold|
|   Pryce Cowdrey|             546.33|     gold|
| Jehanna Priditt|             341.32|     gold|
|   Dael Dearness|             490.75|     gold|
|     Flynn Finch|             596.31|     gold|
+----------------+-------------------+---------+
only showing top 10 rows


In [648]:
# Exibindo somente clientes silver
df_final.filter(col("categoria") == "silver").show(10)

+-------------------+-------------------+---------+
|               name|total_gasto_cliente|categoria|
+-------------------+-------------------+---------+
|     Pierette Sleit|             280.28|   silver|
|  Daisie McAllester|             124.43|   silver|
|       Con Sandifer|             201.18|   silver|
|       Letta Blemen|             292.68|   silver|
|      Brittne Weeds|              85.49|   silver|
|Christos Filipputti|             119.44|   silver|
|    Flint Dabrowski|             200.52|   silver|
| Millisent O'Reagan|             245.62|   silver|
|   Reinhold Brugman|             290.04|   silver|
|   Mitch Kilpatrick|             244.39|   silver|
+-------------------+-------------------+---------+
only showing top 10 rows


## Questão 7

**Enunciado:**

Salvar um arquivo parquet com os três produtos mais consumidos de cada categoria do cliente.

**Resolução:**

In [649]:
# Preparando os dados
df_produtos_por_categoria = df_sales.join(df_classificado, "client_id", "inner") \
    .join(df_products, "product_id", "inner") \
    .groupBy("categoria", "product") \
    .agg(sum("qtde").alias("quantidade_total"))

In [650]:
# Criando a Window Function para ranquear os produtos dentro de cada categoria
window_spec = Window.partitionBy("categoria").orderBy(col("quantidade_total").desc())

In [651]:
# Aplicando o ranking e filtrando apenas os 3 primeiros
df_top3_parquet = df_produtos_por_categoria.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 3) \
    .drop("rank")

In [652]:
# Salvando arquivo final em formato parquet
df_top3_parquet.write.mode("overwrite").parquet("top3_produtos_por_categoria.parquet")

print("Arquivo Parquet salvo com sucesso!")
df_top3_parquet.show(truncate= False)

Arquivo Parquet salvo com sucesso!
+---------+---------------------+----------------+
|categoria|product              |quantidade_total|
+---------+---------------------+----------------+
|gold     |Casaco de Inverno    |26              |
|gold     |Cinto de Couro       |26              |
|gold     |Camisa Xadrez        |23              |
|platinum |Blusa de Manga Longa |57              |
|platinum |Mochila de Lona      |52              |
|platinum |Calça Cintura Alta   |45              |
|silver   |Macaquinho           |7               |
|silver   |Sapatos de Salto Alto|6               |
|silver   |Vestido Midi         |5               |
+---------+---------------------+----------------+

